# 02 — YOLO Data Preparation

This notebook converts the original BDD100K object detection annotations into YOLO format and prepares the dataset for model training.

### Objectives
- Select the object detection classes
- Convert BDD100K bounding boxes to YOLO format
- Normalize bounding-box coordinates
- Generate YOLO label files
- Prepare train and validation data
- Create the YOLO dataset configuration file

### Project Roadmap
**Dataset Exploration → Data Preparation → YOLO Training → Evaluation → Inference → Driving Risk Analysis**



## 1. Environment Setup

In [1]:
# ==============================
# Libraries
# ==============================

from pathlib import Path
import json
import shutil

import numpy as np
from PIL import Image

## 2. Dataset Paths

In [2]:


# Main BDD100K dataset directory
DATASET_ROOT = Path("../data/bdd100k")

# Image directories
IMAGE_ROOT = DATASET_ROOT / "images" / "100k"
TRAIN_IMAGE_DIR = IMAGE_ROOT / "train"
VAL_IMAGE_DIR = IMAGE_ROOT / "val"

# Original BDD100K annotation directories
LABEL_ROOT = DATASET_ROOT / "labels"
TRAIN_LABEL_DIR = LABEL_ROOT / "train"
VAL_LABEL_DIR = LABEL_ROOT / "val"

# Check that all required directories exist
print("Dataset root:", DATASET_ROOT.exists())
print("Train images:", TRAIN_IMAGE_DIR.exists())
print("Val images:", VAL_IMAGE_DIR.exists())
print("Train labels:", TRAIN_LABEL_DIR.exists())
print("Val labels:", VAL_LABEL_DIR.exists())

Dataset root: True
Train images: True
Val images: True
Train labels: True
Val labels: True


## 3. Object Detection Classes

In [3]:


# BDD100K classes used for YOLO object detection
DETECTION_CLASSES = [
    "car",
    "truck",
    "bus",
    "train",
    "person",
    "rider",
    "bike",
    "motor",
    "traffic light",
    "traffic sign",
]

# Create a mapping from class name to class ID
CLASS_TO_ID = {
    class_name: class_id
    for class_id, class_name in enumerate(DETECTION_CLASSES)
}

# Display the class mapping
for class_name, class_id in CLASS_TO_ID.items():
    print(f"{class_id} -> {class_name}")

0 -> car
1 -> truck
2 -> bus
3 -> train
4 -> person
5 -> rider
6 -> bike
7 -> motor
8 -> traffic light
9 -> traffic sign


## 4. Bounding Box Conversion

In [4]:


def convert_bbox_to_yolo(box, image_width, image_height):
    """
    Convert BDD100K bounding box coordinates
    from (x1, y1, x2, y2) to normalized YOLO format.
    """

    # Read original bounding box coordinates
    x1 = box["x1"]
    y1 = box["y1"]
    x2 = box["x2"]
    y2 = box["y2"]

    # Calculate bounding box center
    x_center = (x1 + x2) / 2
    y_center = (y1 + y2) / 2

    # Calculate bounding box width and height
    box_width = x2 - x1
    box_height = y2 - y1

    # Normalize values between 0 and 1
    x_center /= image_width
    y_center /= image_height
    box_width /= image_width
    box_height /= image_height

    return x_center, y_center, box_width, box_height

## 5. YOLO Label Generation

In [11]:
def convert_annotation_to_yolo(annotation_path, image_width=1280, image_height=720):
    """
    Convert one BDD100K annotation file into YOLO label lines.
    """

    # Read the BDD100K JSON annotation
    with open(annotation_path, "r") as f:
        annotation = json.load(f)

    # Get all annotated objects from the image
    objects = annotation["frames"][0]["objects"]

    # Store the final YOLO label lines
    yolo_labels = []

    # Process each annotated object
    for obj in objects:

        category = obj["category"]

        # Keep only the 10 object detection classes
        if category not in DETECTION_CLASSES:
            continue

        # Skip objects without a bounding box
        if "box2d" not in obj:
            continue

        # Convert class name to class ID
        class_id = CLASS_TO_ID[category]

        # Convert bounding box to normalized YOLO format
        x_center, y_center, box_width, box_height = convert_bbox_to_yolo(
            obj["box2d"],
            image_width,
            image_height
        )

        # Skip invalid bounding boxes
        if not is_valid_yolo_box(
            x_center,
            y_center,
            box_width,
            box_height
        ):
            continue

        # Create one YOLO label line
        yolo_line = (
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_width:.6f} "
            f"{box_height:.6f}"
        )

        # Add the converted label to the final list
        yolo_labels.append(yolo_line)

    return yolo_labels

In [12]:

# Bounding Box Validation


def is_valid_yolo_box(x_center, y_center, box_width, box_height):
    """
    Check whether YOLO bounding-box values are valid.
    """

    # Center coordinates must be inside the image
    center_is_valid = (
        0 <= x_center <= 1
        and 0 <= y_center <= 1
    )

    # Width and height must be positive and not larger than the image
    size_is_valid = (
        0 < box_width <= 1
        and 0 < box_height <= 1
    )

    return center_is_valid and size_is_valid

## 6. Generate YOLO Labels for Train and Validation

In [13]:

# Generate YOLO Labels


# Output directories for YOLO labels
YOLO_ROOT = Path("../data/bdd100k_yolo")
YOLO_TRAIN_LABEL_DIR = YOLO_ROOT / "labels" / "train"
YOLO_VAL_LABEL_DIR = YOLO_ROOT / "labels" / "val"

# Create output directories
YOLO_TRAIN_LABEL_DIR.mkdir(parents=True, exist_ok=True)
YOLO_VAL_LABEL_DIR.mkdir(parents=True, exist_ok=True)


def generate_yolo_labels(source_label_dir, output_label_dir):
    """
    Convert all BDD100K JSON annotation files in one split
    into YOLO .txt label files.
    """

    # Get all JSON annotation files
    annotation_files = list(source_label_dir.glob("*.json"))

    # Counters for summary
    converted_files = 0
    total_objects = 0

    # Process each annotation file
    for annotation_path in annotation_files:

        # Convert annotation to YOLO label lines
        yolo_labels = convert_annotation_to_yolo(annotation_path)

        # Create corresponding YOLO .txt filename
        output_path = output_label_dir / f"{annotation_path.stem}.txt"

        # Save YOLO labels
        with open(output_path, "w") as f:
            for label in yolo_labels:
                f.write(label + "\n")

        # Update counters
        converted_files += 1
        total_objects += len(yolo_labels)

    return converted_files, total_objects

In [14]:

# Generate Training Labels

# Convert all training annotations to YOLO format
train_files, train_objects = generate_yolo_labels(
    TRAIN_LABEL_DIR,
    YOLO_TRAIN_LABEL_DIR
)

# Display conversion summary
print("Training conversion completed.")
print(f"YOLO label files : {train_files:,}")
print(f"YOLO objects     : {train_objects:,}")

Training conversion completed.
YOLO label files : 70,000
YOLO objects     : 1,288,010


In [15]:

# Validate Generated Training Labels


# Expected number of detection objects from Notebook 01
expected_train_objects = 1_288_405

# Calculate how many bounding boxes were skipped
skipped_objects = expected_train_objects - train_objects

# Find empty YOLO label files
empty_label_files = [
    label_path
    for label_path in YOLO_TRAIN_LABEL_DIR.glob("*.txt")
    if label_path.stat().st_size == 0
]

# Display validation summary
print(f"Expected objects : {expected_train_objects:,}")
print(f"Saved objects    : {train_objects:,}")
print(f"Skipped objects  : {skipped_objects:,}")
print(f"Empty label files: {len(empty_label_files):,}")

Expected objects : 1,288,405
Saved objects    : 1,288,010
Skipped objects  : 395
Empty label files: 0


In [16]:

# Inspect Invalid Bounding Boxes


invalid_boxes = []

# Check all training annotations
for annotation_path in TRAIN_LABEL_DIR.glob("*.json"):

    with open(annotation_path, "r") as f:
        annotation = json.load(f)

    objects = annotation["frames"][0]["objects"]

    for obj in objects:

        # Check only our object detection classes
        if obj["category"] not in DETECTION_CLASSES:
            continue

        # Skip annotations without bounding boxes
        if "box2d" not in obj:
            continue

        # Convert bounding box to YOLO format
        yolo_box = convert_bbox_to_yolo(
            obj["box2d"],
            1280,
            720
        )

        # Store invalid boxes
        if not is_valid_yolo_box(*yolo_box):
            invalid_boxes.append({
                "file": annotation_path.name,
                "category": obj["category"],
                "box2d": obj["box2d"],
                "yolo_box": yolo_box
            })

print(f"Invalid boxes found: {len(invalid_boxes):,}")

# Show only the first 5 examples
for item in invalid_boxes[:5]:
    print("\nFile:", item["file"])
    print("Category:", item["category"])
    print("BDD100K box:", item["box2d"])
    print("YOLO box:", item["yolo_box"])

Invalid boxes found: 395

File: 007aeb45-f9f5ac8c.json
Category: traffic sign
BDD100K box: {'x1': 300.606818, 'y1': 521.437142, 'x2': 320.856806, 'y2': 521.437142}
YOLO box: (0.242759228125, 0.7242182527777777, 0.015820303125000025, 0.0)

File: 00d15d58-9197cde5.json
Category: car
BDD100K box: {'x1': 653.877854, 'y1': 432.000194, 'x2': 670.408475, 'y2': 432.000194}
YOLO box: (0.517299347265625, 0.6000002694444445, 0.012914547656249997, 0.0)

File: 01853f47-6975b587.json
Category: car
BDD100K box: {'x1': 506.481678, 'y1': 406.012223, 'x2': 519.981669, 'y2': 406.012223}
YOLO box: (0.40096224492187493, 0.5639058652777778, 0.010546867968750017, 0.0)

File: 024b275b-33674b72.json
Category: person
BDD100K box: {'x1': 219.791281, 'y1': 468.084379, 'x2': 219.791281, 'y2': 480.072995}
YOLO box: (0.17171193828125, 0.6584426208333334, 0.0, 0.016650855555555528)

File: 025f49ef-0f6366f3.json
Category: car
BDD100K box: {'x1': 709.539767, 'y1': 385.207243, 'x2': 722.513068, 'y2': 385.207243}
YOLO bo

In [17]:

# Analyze Invalid Box Reasons


zero_width = 0
zero_height = 0
out_of_bounds = 0

# Analyze the invalid boxes already collected
for item in invalid_boxes:
    x_center, y_center, box_width, box_height = item["yolo_box"]

    # Count boxes with zero or negative width
    if box_width <= 0:
        zero_width += 1

    # Count boxes with zero or negative height
    if box_height <= 0:
        zero_height += 1

    # Count boxes with coordinates outside the normalized image range
    if not (0 <= x_center <= 1 and 0 <= y_center <= 1):
        out_of_bounds += 1

# Display the reasons
print(f"Zero-width boxes : {zero_width:,}")
print(f"Zero-height boxes: {zero_height:,}")
print(f"Out-of-bounds    : {out_of_bounds:,}")

Zero-width boxes : 132
Zero-height boxes: 263
Out-of-bounds    : 0


In [18]:

# Generate Validation Labels


# Convert all validation annotations to YOLO format
val_files, val_objects = generate_yolo_labels(
    VAL_LABEL_DIR,
    YOLO_VAL_LABEL_DIR
)

# Display conversion summary
print("Validation conversion completed.")
print(f"YOLO label files : {val_files:,}")
print(f"YOLO objects     : {val_objects:,}")

Validation conversion completed.
YOLO label files : 10,000
YOLO objects     : 185,526


## 7. YOLO Dataset Structure




In [19]:

# Verify Image-Label Matching


def check_image_label_matching(image_dir, yolo_label_dir):
    """
    Check whether every YOLO label file has a corresponding image.
    """

    # Get image and label filenames without extensions
    image_names = {
        path.stem
        for path in image_dir.glob("*.jpg")
    }

    label_names = {
        path.stem
        for path in yolo_label_dir.glob("*.txt")
    }

    # Find missing matches
    labels_without_images = label_names - image_names
    images_without_labels = image_names - label_names

    # Display summary
    print(f"Images            : {len(image_names):,}")
    print(f"YOLO labels       : {len(label_names):,}")
    print(f"Labels w/o image  : {len(labels_without_images):,}")
    print(f"Images w/o label  : {len(images_without_labels):,}")


print("TRAIN")
check_image_label_matching(
    TRAIN_IMAGE_DIR,
    YOLO_TRAIN_LABEL_DIR
)

print("\nVAL")
check_image_label_matching(
    VAL_IMAGE_DIR,
    YOLO_VAL_LABEL_DIR
)

TRAIN
Images            : 70,000
YOLO labels       : 70,000
Labels w/o image  : 0
Images w/o label  : 0

VAL
Images            : 10,000
YOLO labels       : 10,000
Labels w/o image  : 0
Images w/o label  : 0


In [20]:

# Create Final YOLO Label Structure


# Final YOLO label directories
FINAL_TRAIN_LABEL_DIR = DATASET_ROOT / "labels" / "100k" / "train"
FINAL_VAL_LABEL_DIR = DATASET_ROOT / "labels" / "100k" / "val"

# Create the directories
FINAL_TRAIN_LABEL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_VAL_LABEL_DIR.mkdir(parents=True, exist_ok=True)

print("Train YOLO labels:", FINAL_TRAIN_LABEL_DIR)
print("Val YOLO labels  :", FINAL_VAL_LABEL_DIR)

Train YOLO labels: ..\data\bdd100k\labels\100k\train
Val YOLO labels  : ..\data\bdd100k\labels\100k\val


In [21]:

# Copy Generated YOLO Labels


# Copy training YOLO labels to the final dataset structure
for label_path in YOLO_TRAIN_LABEL_DIR.glob("*.txt"):
    shutil.copy2(
        label_path,
        FINAL_TRAIN_LABEL_DIR / label_path.name
    )

# Copy validation YOLO labels to the final dataset structure
for label_path in YOLO_VAL_LABEL_DIR.glob("*.txt"):
    shutil.copy2(
        label_path,
        FINAL_VAL_LABEL_DIR / label_path.name
    )

# Verify the number of copied files
train_final_count = len(list(FINAL_TRAIN_LABEL_DIR.glob("*.txt")))
val_final_count = len(list(FINAL_VAL_LABEL_DIR.glob("*.txt")))

print(f"Train YOLO labels: {train_final_count:,}")
print(f"Val YOLO labels  : {val_final_count:,}")

Train YOLO labels: 70,000
Val YOLO labels  : 10,000


## 8. Dataset Configuration

In [22]:

# Create YOLO Dataset Configuration


# Path for the YOLO dataset configuration file
YAML_PATH = DATASET_ROOT / "dataset.yaml"

# Dataset configuration
yaml_content = f"""# BDD100K YOLO Dataset Configuration

path: {DATASET_ROOT.resolve().as_posix()}

train: images/100k/train
val: images/100k/val

nc: {len(DETECTION_CLASSES)}

names:
"""

# Add class IDs and class names
for class_id, class_name in enumerate(DETECTION_CLASSES):
    yaml_content += f"  {class_id}: {class_name}\n"

# Save the configuration file
with open(YAML_PATH, "w") as f:
    f.write(yaml_content)

print("dataset.yaml created successfully.")
print("Path:", YAML_PATH)

dataset.yaml created successfully.
Path: ..\data\bdd100k\dataset.yaml


In [23]:
# Display the generated dataset configuration
with open(YAML_PATH, "r") as f:
    print(f.read())

# BDD100K YOLO Dataset Configuration

path: D:/Yasna-ML-Projects/Automotive-YOLO-Object-Detection/data/bdd100k

train: images/100k/train
val: images/100k/val

nc: 10

names:
  0: car
  1: truck
  2: bus
  3: train
  4: person
  5: rider
  6: bike
  7: motor
  8: traffic light
  9: traffic sign



## 9. Final Validation

In [24]:

# Final YOLO Label Validation


def validate_yolo_label_file(label_path):
    """
    Validate one YOLO label file.
    """

    errors = []

    # Read all label lines
    with open(label_path, "r") as f:
        lines = f.readlines()

    for line_number, line in enumerate(lines, start=1):

        # Split YOLO label values
        parts = line.strip().split()

        # Each YOLO label must contain 5 values
        if len(parts) != 5:
            errors.append(
                f"Line {line_number}: expected 5 values, got {len(parts)}"
            )
            continue

        # Parse YOLO values
        class_id = int(parts[0])
        x_center = float(parts[1])
        y_center = float(parts[2])
        box_width = float(parts[3])
        box_height = float(parts[4])

        # Validate class ID
        if not 0 <= class_id < len(DETECTION_CLASSES):
            errors.append(
                f"Line {line_number}: invalid class ID {class_id}"
            )

        # Validate normalized bounding box values
        if not is_valid_yolo_box(
            x_center,
            y_center,
            box_width,
            box_height
        ):
            errors.append(
                f"Line {line_number}: invalid bounding box values"
            )

    return errors

In [25]:

# Validate Sample Label Files


import random

# Get YOLO label files
train_yolo_files = list(FINAL_TRAIN_LABEL_DIR.glob("*.txt"))
val_yolo_files = list(FINAL_VAL_LABEL_DIR.glob("*.txt"))

# Randomly select sample files
sample_files = (
    random.sample(train_yolo_files, 10)
    + random.sample(val_yolo_files, 5)
)

validation_errors = {}

# Validate each selected label file
for label_path in sample_files:
    errors = validate_yolo_label_file(label_path)

    if errors:
        validation_errors[label_path.name] = errors

# Display validation result
print(f"Checked files : {len(sample_files)}")
print(f"Files with errors: {len(validation_errors)}")

if not validation_errors:
    print("All sampled YOLO label files are valid.")

Checked files : 15
Files with errors: 0
All sampled YOLO label files are valid.


## 10. Conclusion

- BDD100K annotations were converted to YOLO format.
- Invalid zero-area bounding boxes were removed.
- Train and validation YOLO labels were generated successfully.
- Image-label matching was verified.
- The dataset configuration file was created.
- The dataset is ready for YOLO training.